# Trabajo Práctico Integrador — Entrega 3
## Ciencia de Datos 2026 | UTN FRC | 5K4 | Grupo 15

---

### Descripción de la entrega

Este notebook implementa el pipeline de **modelado y clasificación** sobre el dataset PAMAP2 procesado en la Entrega 2. Se abordan las tres hipótesis definidas:

- **H1:** Clasificación multiclase del tipo de actividad física (12 clases del protocolo estándar)
- **H2:** Distinción entre actividades de baja movilidad — sitting (2), standing (3), nordic walking (7)
- **H3:** Predicción del nivel de esfuerzo físico (bajo / medio / alto) a partir de señales IMU

El dataset de entrada (`pamap2_limpio.csv`) ya tiene las columnas sensoriales escaladas con **StandardScaler** (aplicado en la Entrega 2). **No se vuelve a escalar en este notebook.**

---

### Integrantes — Grupo 15

| Rol | Nombre | Legajo |
|-----|--------|--------|
| Product Owner | Franco Recalde | 94661 |
| Scrum Master | Emilio Sadir | 96622 |
| dev | Rodríguez, Joaquín Ignacio | 85406 |
| dev | Noto, Claudia Carina | 95215 |
| dev | Monteros, Leandro Ignacio | 90258 |
| dev | Ferraro, Ayelen | 99149 |
| dev | Chiaro, Tiziano | 90043 |
| dev | Bailey, Julián Eduardo | 96032 |

---
# Sección 0 — Setup
Importación de librerías, configuración del entorno (Colab / local), carga del dataset procesado y verificación inicial.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid')

print('Librerías cargadas correctamente.')

### Paso 0.1: Configuración del entorno

La notebook detecta automáticamente si se ejecuta en Google Colab o localmente.
En Colab se monta Google Drive y se asume que el proyecto está en `MyDrive/TP_CD/`.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/TP_CD')
else:
    BASE_PATH = Path('..').resolve()

DATA_PATH    = BASE_PATH / 'data' / 'processed' / 'pamap2_limpio.csv'
FIGURES_PATH = BASE_PATH / 'reports' / 'entrega_3' / 'figures'
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

print(f'Entorno:   {"Google Colab" if IN_COLAB else "Local"}')
print(f'Dataset:   {DATA_PATH}')
print(f'Figuras:   {FIGURES_PATH}')

### Paso 0.2: Carga del dataset y verificación

Se carga el CSV procesado en la Entrega 2. Las columnas sensoriales ya están normalizadas (z-score). Se verifica el shape, los tipos y la ausencia de nulos.

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f'Shape: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'Nulos: {df.isnull().sum().sum()}')
print(f'Actividades presentes ({df["actividad_id"].nunique()}): {sorted(df["actividad_id"].unique())}')
print(f'Sujetos ({df["sujeto_id"].nunique()}): {sorted(df["sujeto_id"].unique())}')
print()

# Anomalía conocida: subject109 tiene muy pocas filas
filas_por_sujeto = df.groupby('sujeto_id').size().rename('filas')
print('Filas por sujeto:')
display(filas_por_sujeto.to_frame())
print()
display(df.dtypes.to_frame('dtype').T)

### Paso 0.3: Constantes y funciones auxiliares

In [ ]:
ACTIVIDAD_MAP = {
    1: 'lying',          2: 'sitting',      3: 'standing',
    4: 'walking',        5: 'running',       6: 'cycling',
    7: 'nordic walking', 12: 'asc. stairs', 13: 'desc. stairs',
    16: 'vacuum clean',  17: 'ironing',      24: 'rope jumping'
}

# Features sensoriales (todo excepto tiempo, actividad_id, sujeto_id)
SENSOR_COLS = [c for c in df.columns if c not in ['tiempo', 'actividad_id', 'sujeto_id']]
# Features IMU (excluye frecuencia_cardiaca — usada como target derivado en H3)
IMU_COLS    = [c for c in SENSOR_COLS if c != 'frecuencia_cardiaca']

print(f'SENSOR_COLS ({len(SENSOR_COLS)}): {SENSOR_COLS[:5]} ...')
print(f'IMU_COLS    ({len(IMU_COLS)}):    {IMU_COLS[:5]} ...')


def plot_cm(y_true, y_pred, labels, label_names, title, filename=None, figsize=(12, 10)):
    """Heatmap de matriz de confusión anotado."""
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_names, yticklabels=label_names,
                ax=ax, linewidths=0.4, cbar=False)
    ax.set_xlabel('Predicho', fontsize=11)
    ax.set_ylabel('Real', fontsize=11)
    ax.set_title(title, fontsize=13)
    plt.tight_layout()
    if filename:
        plt.savefig(FIGURES_PATH / filename, dpi=100, bbox_inches='tight')
    plt.show()


def evaluar(nombre, modelo, X_train, X_test, y_train, y_test, labels, label_names,
            skip_train_acc=False):
    """Retorna (acc_train, acc_test, f1_macro_test) e imprime el classification report.
    skip_train_acc=True cuando el modelo memoriza training data (KNN) y el valor no es útil."""
    if skip_train_acc:
        acc_train = float('nan')
        print(f'=== {nombre} ===')
        print('Accuracy train: N/A (KNN retorna ~100% sobre train — no es informativo)')
    else:
        acc_train = accuracy_score(y_train, modelo.predict(X_train))
        print(f'=== {nombre} ===')
        print(f'Accuracy train: {acc_train:.4f}', end='')

    y_pred   = modelo.predict(X_test)
    acc_test = accuracy_score(y_test, y_pred)
    f1_mac   = f1_score(y_test, y_pred, labels=labels, average='macro', zero_division=0)

    if not skip_train_acc:
        print(f'  |  Accuracy test: {acc_test:.4f}  |  F1 macro: {f1_mac:.4f}')
    else:
        print(f'Accuracy test:  {acc_test:.4f}  |  F1 macro: {f1_mac:.4f}')
    print()
    print(classification_report(y_test, y_pred, labels=labels,
                                target_names=label_names, zero_division=0))
    return acc_train, acc_test, f1_mac

---
# Sección 1 — H1: Clasificación multiclase de actividades

**Hipótesis:** *Es posible clasificar con alta precisión el tipo de actividad física realizada a partir de las señales inerciales y de frecuencia cardíaca.*

- **Target:** `actividad_id` (12 clases del protocolo estándar)
- **Features:** todas las columnas sensoriales escaladas (`SENSOR_COLS`)
- **Modelos:** Random Forest · KNN

### 1.1 — Distribución de clases

Antes de entrenar es fundamental visualizar el balance de clases. Un dataset desbalanceado puede hacer que el modelo aprenda a predecir siempre la clase mayoritaria. Por eso se usa `class_weight='balanced'` en los modelos que lo soportan.

In [ ]:
X_h1 = df[SENSOR_COLS]
y_h1 = df['actividad_id']

act_counts = y_h1.value_counts().sort_index()
act_labels = [ACTIVIDAD_MAP.get(a, str(a)) for a in act_counts.index]

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(act_labels, act_counts.values, color=sns.color_palette('tab20', len(act_counts)))
ax.bar_label(bars, fmt='{:,.0f}', fontsize=8, padding=3)
ax.set_title('H1 — Distribución de registros por actividad')
ax.set_xlabel('Actividad')
ax.set_ylabel('Cantidad de registros')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'h1_distribucion_clases.png', dpi=100, bbox_inches='tight')
plt.show()

print('Conteo por clase:')
print(act_counts.rename(index=ACTIVIDAD_MAP).to_string())

### 1.2 — División train / test

Se aplica **split estratificado** (80 / 20) sobre `actividad_id` para mantener la proporción de clases.

> **Nota sobre data leakage:** El split correcto para HAR es **Leave-One-Subject-Out Cross-Validation (LOSO-CV)**, que entrena con 8 sujetos y evalúa con el 9.º, iterando sobre todos. Esto evita que el modelo «memorice» los patrones de movimiento particulares de cada persona.  
> Para esta entrega se usa split estratificado por simplicidad, pero los resultados deben interpretarse con cautela: el modelo probablemente sobreestima su capacidad de generalizar a nuevas personas.  
> **Limitación adicional:** el sujeto 109 tiene muy pocas filas (~50-80 después del pipeline ETL), lo que introduce ruido estadístico y puede afectar la estimación de performance real.

In [ ]:
X_train_h1, X_test_h1, y_train_h1, y_test_h1 = train_test_split(
    X_h1, y_h1, test_size=0.20, random_state=42, stratify=y_h1
)

print(f'Train: {X_train_h1.shape[0]:,} filas  |  Test: {X_test_h1.shape[0]:,} filas')
print(f'Clases en train: {sorted(y_train_h1.unique())}')
print(f'Clases en test:  {sorted(y_test_h1.unique())}')

### 1.3 — Entrenamiento de modelos

Se entrenan dos clasificadores:

- **Random Forest** (`n_estimators=100`, `class_weight='balanced'`): ensamble de árboles con votación por mayoría. Robusto al ruido y al desbalance de clases. Muy utilizado en literatura HAR.
- **KNN** (`n_neighbors=5`): clasifica por los 5 vecinos más cercanos en el espacio de features. Conceptualmente simple, no hace suposiciones sobre la distribución de los datos. Puede ser lento con datasets grandes.

> **Aviso:** con ~153K filas de entrenamiento, KNN puede tardar varios minutos en la fase de predicción.

In [ ]:
rf_h1 = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                                random_state=42, n_jobs=-1)
rf_h1.fit(X_train_h1, y_train_h1)
print('Random Forest H1 entrenado.')

In [ ]:
knn_h1 = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
knn_h1.fit(X_train_h1, y_train_h1)
print('KNN H1 entrenado.')

### 1.4 — Evaluación: métricas y matrices de confusión

Se reportan **accuracy** en train y test (para detectar overfitting), **F1 macro** (métrica más justa ante desbalance de clases), y la **classification report** completa por clase.  
La **matriz de confusión** permite identificar qué actividades se confunden entre sí.

In [ ]:
h1_labels     = sorted(y_h1.unique())
h1_label_names = [ACTIVIDAD_MAP.get(a, str(a)) for a in h1_labels]

acc_train_rf_h1, acc_test_rf_h1, f1_rf_h1 = evaluar(
    'Random Forest — H1', rf_h1,
    X_train_h1, X_test_h1, y_train_h1, y_test_h1,
    h1_labels, h1_label_names
)

plot_cm(
    y_test_h1, rf_h1.predict(X_test_h1),
    h1_labels, h1_label_names,
    'H1 — Matriz de confusión: Random Forest',
    'h1_cm_rf.png', figsize=(13, 11)
)

In [ ]:
acc_train_knn_h1, acc_test_knn_h1, f1_knn_h1 = evaluar(
    'KNN — H1', knn_h1,
    X_train_h1, X_test_h1, y_train_h1, y_test_h1,
    h1_labels, h1_label_names,
    skip_train_acc=True   # KNN encontraría cada punto como su propio vecino → no informativo
)

plot_cm(
    y_test_h1, knn_h1.predict(X_test_h1),
    h1_labels, h1_label_names,
    'H1 — Matriz de confusión: KNN (k=5)',
    'h1_cm_knn.png', figsize=(13, 11)
)

### 1.5 — Tabla comparativa H1

In [ ]:
resumen_h1 = pd.DataFrame([
    {'Modelo': 'Random Forest', 'Acc. train': acc_train_rf_h1,
     'Acc. test': acc_test_rf_h1, 'F1 macro test': f1_rf_h1},
    {'Modelo': 'KNN (k=5)',     'Acc. train': acc_train_knn_h1,
     'Acc. test': acc_test_knn_h1, 'F1 macro test': f1_knn_h1},
]).set_index('Modelo')

display(resumen_h1.style.format('{:.4f}').highlight_max(axis=0, color='#d4f0d4'))

### 1.6 — Análisis de resultados H1

**Confusiones esperadas y su explicación fisiológica:**

- **Sitting ↔ Standing:** ambas son actividades sedentarias con señales IMU muy similares. La diferencia principal es la posición del torso, que el acelerómetro del pecho puede capturar, pero la señal es sutil.
- **Walking ↔ Nordic walking:** el patrón de pasos es prácticamente idéntico; nordic walking agrega movimiento de brazos con bastones. El giroscopio de la mano (mano_giro_x/y/z) es la señal discriminante.
- **Ascending ↔ Descending stairs:** el patrón de pasos es muy parecido. La diferencia radica en la inclinación del torso y la fase de impacto en el tobillo.
- **Lying:** actividad bien separada — la aceleración gravitacional proyectada en los tres ejes es muy distinta a cualquier actividad en bipedestación.

**Sobre el overfitting en Random Forest:** si la accuracy de train es significativamente mayor que la de test, el modelo memorizó patrones específicos de los sujetos vistos durante el entrenamiento, lo que refuerza la necesidad de LOSO-CV en entregas futuras.

---
# Sección 2 — H2: Distinguir actividades de baja movilidad

**Hipótesis:** *Las señales del acelerómetro permiten distinguir entre actividades de baja movilidad corporal — sitting, standing y nordic walking — que a simple vista parecen similares desde el punto de vista inercial.*

- **Target:** `actividad_id` ∈ {2 (sitting), 3 (standing), 7 (nordic walking)}
- **Features:** todas las columnas sensoriales escaladas (`SENSOR_COLS`)
- **Modelos:** SVM · Random Forest

### 2.1 — Filtrado del subconjunto y distribución de clases

Se trabaja sobre el subconjunto de 3 actividades. Nordic walking es dinámicamente diferente a sitting/standing, lo que introduce asimetría: H2 tiene tanto un par difícil (sitting vs standing) como uno más fácil (ambas vs nordic walking).

In [ ]:
H2_ACTS = [2, 3, 7]
df_h2 = df[df['actividad_id'].isin(H2_ACTS)].copy()

X_h2 = df_h2[SENSOR_COLS]
y_h2 = df_h2['actividad_id']

print(f'Subconjunto H2: {len(df_h2):,} filas')
print()
cnt_h2 = y_h2.value_counts().sort_index()
names_h2 = [ACTIVIDAD_MAP[a] for a in cnt_h2.index]
print(cnt_h2.rename(index=ACTIVIDAD_MAP).to_string())

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(names_h2, cnt_h2.values, color=['#4878cf', '#6acc65', '#d65f5f'])
ax.bar_label(bars, fmt='{:,.0f}', padding=3)
ax.set_title('H2 — Distribución de clases (subconjunto)')
ax.set_ylabel('Registros')
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'h2_distribucion_clases.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
X_train_h2, X_test_h2, y_train_h2, y_test_h2 = train_test_split(
    X_h2, y_h2, test_size=0.20, random_state=42, stratify=y_h2
)

print(f'Train: {X_train_h2.shape[0]:,}  |  Test: {X_test_h2.shape[0]:,}')

### 2.2 — Modelos

- **SVM** (`kernel='rbf'`, `class_weight='balanced'`, `C=1.0`): construye un hiperplano de máximo margen en un espacio de alta dimensión. El kernel RBF captura fronteras no lineales, útil cuando las actividades no son linealmente separables en el espacio de features.
- **Random Forest** (`n_estimators=100`, `class_weight='balanced'`): mismo que en H1, para comparación directa.

> **Aviso:** SVM con kernel RBF puede tardar varios minutos en conjuntos de datos grandes (>30K filas de entrenamiento). Es computacionalmente más costoso que Random Forest.

In [ ]:
svm_h2 = SVC(kernel='rbf', class_weight='balanced', C=1.0, random_state=42)
svm_h2.fit(X_train_h2, y_train_h2)
print('SVM H2 entrenado.')

In [ ]:
rf_h2 = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                                random_state=42, n_jobs=-1)
rf_h2.fit(X_train_h2, y_train_h2)
print('Random Forest H2 entrenado.')

### 2.3 — Evaluación H2

In [ ]:
h2_labels     = H2_ACTS
h2_label_names = [ACTIVIDAD_MAP[a] for a in h2_labels]

acc_train_svm_h2, acc_test_svm_h2, f1_svm_h2 = evaluar(
    'SVM (RBF) — H2', svm_h2,
    X_train_h2, X_test_h2, y_train_h2, y_test_h2,
    h2_labels, h2_label_names
)
plot_cm(
    y_test_h2, svm_h2.predict(X_test_h2),
    h2_labels, h2_label_names,
    'H2 — Matriz de confusión: SVM (RBF)',
    'h2_cm_svm.png', figsize=(7, 6)
)

acc_train_rf_h2, acc_test_rf_h2, f1_rf_h2 = evaluar(
    'Random Forest — H2', rf_h2,
    X_train_h2, X_test_h2, y_train_h2, y_test_h2,
    h2_labels, h2_label_names
)
plot_cm(
    y_test_h2, rf_h2.predict(X_test_h2),
    h2_labels, h2_label_names,
    'H2 — Matriz de confusión: Random Forest',
    'h2_cm_rf.png', figsize=(7, 6)
)

resumen_h2 = pd.DataFrame([
    {'Modelo': 'SVM (RBF)',    'Acc. train': acc_train_svm_h2,
     'Acc. test': acc_test_svm_h2, 'F1 macro test': f1_svm_h2},
    {'Modelo': 'Random Forest', 'Acc. train': acc_train_rf_h2,
     'Acc. test': acc_test_rf_h2, 'F1 macro test': f1_rf_h2},
]).set_index('Modelo')
display(resumen_h2.style.format('{:.4f}').highlight_max(axis=0, color='#d4f0d4'))

### 2.4 — Análisis de resultados H2

**¿Por qué H2 es más difícil que H1 para el par sitting/standing?**

En H1, el modelo tiene 12 clases con señales muy diversas (running, rope jumping, lying) que anclan el espacio de decisión. Sitting y standing comparten casi el mismo estado estático: la gravedad se proyecta de forma muy similar en los tres ejes de cada IMU, y la frecuencia cardíaca es prácticamente idéntica. Las diferencias son sutiles: la orientación del torso (pecho) y la distribución del peso en el tobillo.

Nordic walking, por el contrario, es distinguible fácilmente: tiene movimiento rítmico de piernas y brazos. La confusión se concentrará casi exclusivamente en el par sitting ↔ standing.

**SVM vs Random Forest en este contexto:** SVM puede encontrar fronteras no lineales más ajustadas en el espacio de alta dimensión (40 features), lo que puede ser ventajoso cuando las clases se superponen parcialmente.

---
# Sección 3 — H3: Predicción del nivel de esfuerzo desde señales IMU

**Hipótesis:** *La frecuencia cardíaca puede agruparse en niveles de esfuerzo físico (bajo/medio/alto) y dichos niveles son predecibles a partir de las señales inerciales de las IMU.*

- **Feature engineering:** `nivel_esfuerzo` derivado de `frecuencia_cardiaca` (percentiles 33/66)
- **Target:** `nivel_esfuerzo` (3 clases ordinales)
- **Features:** columnas IMU escaladas (`IMU_COLS`) — `frecuencia_cardiaca` **excluida** (es la fuente del target: usarla sería data leakage directo)
- **Modelos:** Decision Tree · Random Forest

### 3.1 — Feature engineering: `nivel_esfuerzo`

Se crean tres categorías balanceadas por cuantil:
- **bajo:** `frecuencia_cardiaca` ≤ percentil 33
- **medio:** percentil 33 < `frecuencia_cardiaca` ≤ percentil 66
- **alto:** `frecuencia_cardiaca` > percentil 66

Dado que `frecuencia_cardiaca` ya está escalada con z-score en el CSV, los umbrales serán valores z, no BPM. Esto es correcto: la clasificación por cuantil es independiente de la escala original.

In [ ]:
q33 = df['frecuencia_cardiaca'].quantile(0.33)
q66 = df['frecuencia_cardiaca'].quantile(0.66)

print(f'Umbral bajo/medio  (percentil 33): {q33:.4f}')
print(f'Umbral medio/alto  (percentil 66): {q66:.4f}')

df['nivel_esfuerzo'] = pd.cut(
    df['frecuencia_cardiaca'],
    bins=[-np.inf, q33, q66, np.inf],
    labels=['bajo', 'medio', 'alto']
).astype(str)   # str para compatibilidad con sklearn metrics

print()
print('Distribución de nivel_esfuerzo:')
orden = ['bajo', 'medio', 'alto']
print(df['nivel_esfuerzo'].value_counts().reindex(orden).to_string())

fig, ax = plt.subplots(figsize=(6, 4))
cnt_h3 = df['nivel_esfuerzo'].value_counts().reindex(orden)
bars = ax.bar(cnt_h3.index, cnt_h3.values, color=['#2196F3', '#FF9800', '#F44336'])
ax.bar_label(bars, fmt='{:,.0f}', padding=3)
ax.set_title('H3 — Distribución de nivel de esfuerzo')
ax.set_ylabel('Registros')
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'h3_distribucion_esfuerzo.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
X_h3 = df[IMU_COLS]
y_h3 = df['nivel_esfuerzo']

X_train_h3, X_test_h3, y_train_h3, y_test_h3 = train_test_split(
    X_h3, y_h3, test_size=0.20, random_state=42, stratify=y_h3
)

print(f'Train: {X_train_h3.shape[0]:,}  |  Test: {X_test_h3.shape[0]:,}')
print(f'Features IMU: {X_h3.shape[1]} columnas (frecuencia_cardiaca excluida)')

### 3.2 — Modelos

- **Decision Tree** (`max_depth=10`, `class_weight='balanced'`): modelo interpretable que genera reglas de decisión explícitas. `max_depth=10` limita el overfitting.
- **Random Forest** (`n_estimators=100`, `class_weight='balanced'`): mejora la estabilidad del árbol individual mediante el promedio de 100 árboles distintos.

In [ ]:
dt_h3 = DecisionTreeClassifier(max_depth=10, class_weight='balanced', random_state=42)
dt_h3.fit(X_train_h3, y_train_h3)
print('Decision Tree H3 entrenado.')

In [ ]:
rf_h3 = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                                random_state=42, n_jobs=-1)
rf_h3.fit(X_train_h3, y_train_h3)
print('Random Forest H3 entrenado.')

### 3.3 — Evaluación H3

In [ ]:
h3_labels     = ['bajo', 'medio', 'alto']

acc_train_dt_h3, acc_test_dt_h3, f1_dt_h3 = evaluar(
    'Decision Tree — H3', dt_h3,
    X_train_h3, X_test_h3, y_train_h3, y_test_h3,
    h3_labels, h3_labels
)
plot_cm(
    y_test_h3, dt_h3.predict(X_test_h3),
    h3_labels, h3_labels,
    'H3 — Matriz de confusión: Decision Tree',
    'h3_cm_dt.png', figsize=(7, 6)
)

acc_train_rf_h3, acc_test_rf_h3, f1_rf_h3 = evaluar(
    'Random Forest — H3', rf_h3,
    X_train_h3, X_test_h3, y_train_h3, y_test_h3,
    h3_labels, h3_labels
)
plot_cm(
    y_test_h3, rf_h3.predict(X_test_h3),
    h3_labels, h3_labels,
    'H3 — Matriz de confusión: Random Forest',
    'h3_cm_rf.png', figsize=(7, 6)
)

resumen_h3 = pd.DataFrame([
    {'Modelo': 'Decision Tree', 'Acc. train': acc_train_dt_h3,
     'Acc. test': acc_test_dt_h3, 'F1 macro test': f1_dt_h3},
    {'Modelo': 'Random Forest', 'Acc. train': acc_train_rf_h3,
     'Acc. test': acc_test_rf_h3, 'F1 macro test': f1_rf_h3},
]).set_index('Modelo')
display(resumen_h3.style.format('{:.4f}').highlight_max(axis=0, color='#d4f0d4'))

### 3.4 — Visualización del árbol de decisión (primeros 3 niveles)

Se visualizan los primeros 3 niveles del árbol para hacer legibles las reglas de decisión aprendidas. Cada nodo muestra la feature, el umbral de corte, la impureza y la distribución de clases.

In [ ]:
fig, ax = plt.subplots(figsize=(22, 8))
plot_tree(
    dt_h3, ax=ax, max_depth=3,
    feature_names=IMU_COLS, class_names=h3_labels,
    filled=True, rounded=True, fontsize=8
)
ax.set_title('H3 — Decision Tree (primeros 3 niveles)', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'h3_decision_tree.png', dpi=100, bbox_inches='tight')
plt.show()

### 3.5 — Feature importances del Random Forest (top 15)

Las importancias de features del Random Forest indican cuánto contribuye cada señal IMU a la predicción del nivel de esfuerzo. Señales con alta importancia son las más discriminativas para clasificar entre bajo / medio / alto esfuerzo.

In [ ]:
importances = pd.Series(rf_h3.feature_importances_, index=IMU_COLS)
top15 = importances.nlargest(15).sort_values()

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(top15.index, top15.values, color=sns.color_palette('viridis', len(top15)))
ax.set_xlabel('Importancia relativa')
ax.set_title('H3 — Top 15 features por importancia (Random Forest)')
for bar, v in zip(bars, top15.values):
    ax.text(v + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{v:.4f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'h3_feature_importances.png', dpi=100, bbox_inches='tight')
plt.show()

print('Top 15 features:')
print(top15.sort_values(ascending=False).round(4).to_string())

### 3.6 — Análisis de resultados H3

**¿Qué señales predicen mejor el nivel de esfuerzo?**

Se espera que las señales de aceleración del tobillo (`tobillo_acel1_*`) y del pecho (`pecho_acel1_*`) sean las más importantes. El tobillo captura la intensidad del paso y la cadencia, mientras que el pecho refleja el movimiento del centro de masa. A mayor intensidad de movimiento → mayor esfuerzo cardiovascular.

La temperatura (`*_temperatura`) puede también aportar: el calor corporal aumenta con la intensidad del ejercicio, aunque con un retraso temporal.

**Confusión esperada: bajo ↔ medio**

Las clases se definen por cuantiles de frecuencia cardíaca, no por actividades discretas. Esto implica que la frontera entre bajo y medio es difusa desde el punto de vista de las señales IMU: una caminata lenta puede estar en nivel "medio" si el sujeto tiene frecuencia cardíaca basal alta, pero las IMU no detectan esa información fisiológica. Es el límite intrínseco de H3: predecir la respuesta cardíaca solo desde el movimiento.

**Limitación adicional:** las clases bajo/medio/alto se definen globalmente sobre todos los sujetos. Sujetos con diferente condición física tienen respuestas cardíacas distintas para el mismo movimiento, lo que introduce ruido en los labels.

---
# Sección 4 — Resumen y conclusiones

### 4.1 — Tabla comparativa de los tres problemas

In [ ]:
resumen_global = pd.DataFrame([
    {'Hipótesis': 'H1 — Multiclase (12 act.)', 'Modelo': 'Random Forest',
     'Acc. train': acc_train_rf_h1, 'Acc. test': acc_test_rf_h1, 'F1 macro test': f1_rf_h1},
    {'Hipótesis': 'H1 — Multiclase (12 act.)', 'Modelo': 'KNN (k=5)',
     'Acc. train': acc_train_knn_h1, 'Acc. test': acc_test_knn_h1, 'F1 macro test': f1_knn_h1},
    {'Hipótesis': 'H2 — Baja movilidad (3 act.)', 'Modelo': 'SVM (RBF)',
     'Acc. train': acc_train_svm_h2, 'Acc. test': acc_test_svm_h2, 'F1 macro test': f1_svm_h2},
    {'Hipótesis': 'H2 — Baja movilidad (3 act.)', 'Modelo': 'Random Forest',
     'Acc. train': acc_train_rf_h2, 'Acc. test': acc_test_rf_h2, 'F1 macro test': f1_rf_h2},
    {'Hipótesis': 'H3 — Nivel esfuerzo (3 niv.)', 'Modelo': 'Decision Tree',
     'Acc. train': acc_train_dt_h3, 'Acc. test': acc_test_dt_h3, 'F1 macro test': f1_dt_h3},
    {'Hipótesis': 'H3 — Nivel esfuerzo (3 niv.)', 'Modelo': 'Random Forest',
     'Acc. train': acc_train_rf_h3, 'Acc. test': acc_test_rf_h3, 'F1 macro test': f1_rf_h3},
]).set_index(['Hipótesis', 'Modelo'])

print('TABLA COMPARATIVA — PIPELINE DE MODELADO PAMAP2')
display(resumen_global.style.format('{:.4f}').highlight_max(
    subset=['Acc. test', 'F1 macro test'], axis=0, color='#d4f0d4'
))

### 4.2 — Conclusiones

**¿Cuál hipótesis resultó más fácil/difícil de predecir?**

- **H1** es el problema más rico en señal: 12 actividades con patrones de movimiento muy distintos (lying, running, rope jumping). Los sensores IMU capturan diferencias claras entre la mayoría de las clases. La confusión se concentra en el par sitting/standing y en la dupla ascending/descending stairs.

- **H2** es conceptualmente más difícil para el par sitting/standing: la diferencia entre estar sentado y de pie es sutil desde el punto de vista IMU. Nordic walking es fácilmente distinguible, lo que "salva" la accuracy general del subproblema.

- **H3** es el más desafiante en términos conceptuales: se intenta predecir una respuesta fisiológica (frecuencia cardíaca) a partir de señales de movimiento. La relación no es directa — depende de la condición física individual de cada sujeto, y el label se define globalmente. Aun así, señales de alta actividad motora deberían correlacionar con niveles altos de esfuerzo.

---

**Limitaciones del split actual vs LOSO-CV**

El split estratificado por clase (80/20) incluye filas del mismo sujeto en train y test. Esto introduce **data leakage inter-sujeto**: el modelo aprende patrones específicos de cada persona (su cadencia de pasos, su rango de frecuencia cardíaca basal, su estilo de movimiento) y los re-encuentra en el test set.

**Leave-One-Subject-Out CV** entrena con 8 sujetos y evalúa con el 9.º, iterando 9 veces. Mide la capacidad del modelo de generalizar a **personas no vistas**, que es el escenario real de deployment de un sistema HAR.

**Anomalía de subject109:** este sujeto tiene apenas ~50-80 filas en el dataset final (vs ~20.000 del resto). Su escasa representación hace que sus filas en el split de test no sean estadísticamente significativas, y en LOSO-CV sería el fold más débil. Se recomienda investigar si su archivo `.dat` está completo o es un sujeto parcial.

---

**Próximos pasos (Entrega 4 — Data Storytelling)**

- Implementar LOSO-CV para obtener métricas de generalización reales
- Explorar optimización de hiperparámetros (GridSearchCV / RandomizedSearchCV)
- Analizar curvas ROC y PR para cada clase
- Comparar modelos adicionales: Gradient Boosting, LSTM (dado el caracter temporal de la señal)
- Construir el dashboard de presentación con las visualizaciones clave del proyecto